# Trial Generation Pipeline — v8

## What changed from v5

Identical task design and generation. The one addition: the **test set now records the two intensities on every localisation-conflict trial** (`aud_int`, `vis_int`), so accuracy can be broken down by how much the stronger cue wins by. Everything else (noise, labels, balance, the random-label train / stronger-wins test split) is unchanged.

Single 4-class output: 0 = no detection, 1 = detection, 2 = right, 3 = left. Channel order: aud_L, aud_R, vis_L, vis_R.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)
OUT_DIR = Path("./generated_trials_v8")
OUT_DIR.mkdir(exist_ok=True)
print("Setup complete.")

## 2. Parameters

In [ ]:
# Temporal structure
TRIAL_DURATION_S = 2.5
STIM_ONSET_S = 0.5
STIM_OFFSET_S = 1.0
TIMESTEP_S = 0.05
N_TIMESTEPS = int(TRIAL_DURATION_S / TIMESTEP_S)
STIM_ONSET_T = int(STIM_ONSET_S / TIMESTEP_S)
STIM_OFFSET_T = int(STIM_OFFSET_S / TIMESTEP_S)

# Channels
N_CHANNELS = 4
CH_AUD_L, CH_AUD_R, CH_VIS_L, CH_VIS_R = 0, 1, 2, 3

# Intensity
BASELINE_NOISE_STD = 0.1
STIM_INTENSITY_MIN = 1.0
STIM_INTENSITY_MAX = 3.0

# Training counts (balanced to 1200 per class; det_multisensory NOT trained).
# class 0: det_absent(600) + det_visual_only(600)             = 1200
# class 1: det_auditory_only(1200)                            = 1200
# class 2: 3 loc-right types(300 each) + 2 conflicts*150      = 1200
# class 3: 3 loc-left  types(300 each) + 2 conflicts*150      = 1200
N_TRAIN_ABSENT = 600
N_TRAIN_VIS_DET = 600
N_TRAIN_AUD_DET = 1200
N_TRAIN_LOC_PER_TYPE = 300
N_TRAIN_CONFLICT_PER_TYPE = 300
N_TEST_PER_TYPE = 100

print("Timesteps:", N_TIMESTEPS, " stim window:", STIM_ONSET_T, "-", STIM_OFFSET_T)
total_train = (N_TRAIN_ABSENT + N_TRAIN_VIS_DET + N_TRAIN_AUD_DET
               + 6*N_TRAIN_LOC_PER_TYPE + 2*N_TRAIN_CONFLICT_PER_TYPE)
print("Total training trials:", total_train, " test trials:", 12*N_TEST_PER_TYPE)

## 3. Explicit subtask table (binary inputs and binary outputs)

Output shown as one-hot over [noDet, det, right, left]; the network uses a single 4-class softmax.

| # | Subtask | aL | aR | vL | vR | class | noDet | det | R | L | train? | group |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 1 | det_absent | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | yes | det |
| 2 | det_auditory_only | 1 | 1 | 0 | 0 | 1 | 0 | 1 | 0 | 0 | yes | det |
| 3 | det_visual_only | 0 | 0 | 1 | 1 | 0 | 1 | 0 | 0 | 0 | yes | det |
| 4 | det_multisensory | 1 | 1 | 1 | 1 | random 0/1 | - | - | - | - | **test only** | det |
| 5 | loc_auditory_only_L | 1 | 0 | 0 | 0 | 3 | 0 | 0 | 0 | 1 | yes | loc |
| 6 | loc_auditory_only_R | 0 | 1 | 0 | 0 | 2 | 0 | 0 | 1 | 0 | yes | loc |
| 7 | loc_visual_only_L | 0 | 0 | 1 | 0 | 3 | 0 | 0 | 0 | 1 | yes | loc |
| 8 | loc_visual_only_R | 0 | 0 | 0 | 1 | 2 | 0 | 0 | 1 | 0 | yes | loc |
| 9 | loc_multisensory_same_L | 1 | 0 | 1 | 0 | 3 | 0 | 0 | 0 | 1 | yes | loc |
| 10 | loc_multisensory_same_R | 0 | 1 | 0 | 1 | 2 | 0 | 0 | 1 | 0 | yes | loc |
| 11 | loc_conflict_audL_visR | 1 | 0 | 0 | 1 | random 2/3 | 0 | 0 | half | half | yes | loc |
| 12 | loc_conflict_audR_visL | 0 | 1 | 1 | 0 | random 2/3 | 0 | 0 | half | half | yes | loc |

`det_multisensory` is the detection conflict: never trained, only tested.

## 4. Core trial generation

In [ ]:
def generate_trial(per_channel_intensities, rng):
    # Baseline Gaussian noise on all channels; add intensity on active channels during stimulus window.
    X = rng.normal(loc=0.0, scale=BASELINE_NOISE_STD, size=(N_CHANNELS, N_TIMESTEPS))
    for ch, intens in per_channel_intensities.items():
        X[ch, STIM_ONSET_T:STIM_OFFSET_T] += intens
    return X

## 5. Non-conflict subtasks (corrected labels)

In [ ]:
# (active channels, 4-class label). det_visual_only is now class 0.
NON_CONFLICT_SPECS = {
    "det_absent":              ([],                   0),
    "det_auditory_only":       ([CH_AUD_L, CH_AUD_R], 1),
    "det_visual_only":         ([CH_VIS_L, CH_VIS_R], 0),
    "loc_auditory_only_L":     ([CH_AUD_L],           3),
    "loc_auditory_only_R":     ([CH_AUD_R],           2),
    "loc_visual_only_L":       ([CH_VIS_L],           3),
    "loc_visual_only_R":       ([CH_VIS_R],           2),
    "loc_multisensory_same_L": ([CH_AUD_L, CH_VIS_L], 3),
    "loc_multisensory_same_R": ([CH_AUD_R, CH_VIS_R], 2),
}

def make_non_conflict_trial(trial_type, rng):
    channels, label = NON_CONFLICT_SPECS[trial_type]
    intensity = rng.uniform(STIM_INTENSITY_MIN, STIM_INTENSITY_MAX)
    return generate_trial({c: intensity for c in channels}, rng), label

def make_detection_conflict_trial(rng):
    # det_multisensory: all four channels at one matched-salience intensity. Test only.
    # Nominal label 1 = "detected" so accuracy on this row reads as auditory-dominance rate.
    intensity = rng.uniform(STIM_INTENSITY_MIN, STIM_INTENSITY_MAX)
    return generate_trial({CH_AUD_L: intensity, CH_AUD_R: intensity,
                           CH_VIS_L: intensity, CH_VIS_R: intensity}, rng), 1

## 6. Localisation conflict trials

In [ ]:
def make_conflict_trial_random_label(trial_type, label, rng):
    # TRAIN: independent random intensities, externally provided label (forced 50/50).
    ia = rng.uniform(STIM_INTENSITY_MIN, STIM_INTENSITY_MAX)
    iv = rng.uniform(STIM_INTENSITY_MIN, STIM_INTENSITY_MAX)
    if trial_type == "loc_conflict_audL_visR":
        X = generate_trial({CH_AUD_L: ia, CH_VIS_R: iv}, rng)
    else:
        X = generate_trial({CH_AUD_R: ia, CH_VIS_L: iv}, rng)
    return X, label

def make_conflict_trial_stronger_wins(trial_type, audio_wins, rng):
    # TEST: label follows the stronger modality. Also returns the auditory and visual
    # intensities so the conflict can be analysed by intensity gap (v8 addition).
    a = rng.uniform(STIM_INTENSITY_MIN, STIM_INTENSITY_MAX)
    b = rng.uniform(STIM_INTENSITY_MIN, STIM_INTENSITY_MAX)
    hi, lo = max(a, b), min(a, b)
    ia, iv = (hi, lo) if audio_wins else (lo, hi)
    if trial_type == "loc_conflict_audL_visR":
        X = generate_trial({CH_AUD_L: ia, CH_VIS_R: iv}, rng)
        label = 3 if audio_wins else 2   # aud-left wins -> left(3); vis-right wins -> right(2)
    else:
        X = generate_trial({CH_AUD_R: ia, CH_VIS_L: iv}, rng)
        label = 2 if audio_wins else 3   # aud-right wins -> right(2); vis-left wins -> left(3)
    return X, label, ia, iv

## 7. Build the training set (11 subtasks; det_multisensory excluded)

In [ ]:
def build_train_set(rng):
    X_list, y_list, t_list = [], [], []
    for t, n in [("det_absent", N_TRAIN_ABSENT),
                 ("det_visual_only", N_TRAIN_VIS_DET),
                 ("det_auditory_only", N_TRAIN_AUD_DET)]:
        for _ in range(n):
            X, y = make_non_conflict_trial(t, rng)
            X_list.append(X); y_list.append(y); t_list.append(t)
    for t in ["loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
              "loc_multisensory_same_L","loc_multisensory_same_R"]:
        for _ in range(N_TRAIN_LOC_PER_TYPE):
            X, y = make_non_conflict_trial(t, rng)
            X_list.append(X); y_list.append(y); t_list.append(t)
    for t in ["loc_conflict_audL_visR","loc_conflict_audR_visL"]:
        n_left = N_TRAIN_CONFLICT_PER_TYPE // 2
        labels = [3]*n_left + [2]*(N_TRAIN_CONFLICT_PER_TYPE - n_left)
        labels = list(rng.permutation(labels))
        for lb in labels:
            X, y = make_conflict_trial_random_label(t, lb, rng)
            X_list.append(X); y_list.append(y); t_list.append(t)
    idx = rng.permutation(len(X_list))
    return (np.array(X_list)[idx].astype(np.float32),
            np.array(y_list)[idx].astype(np.int64),
            np.array(t_list)[idx])

print("Building training set...")
X_train, y_train, types_train = build_train_set(rng)
print("Training set:", X_train.shape, " total:", len(y_train))

## 8. Build the test set (all 12 subtasks)

In [ ]:
def build_test_set(rng):
    X_list, y_list, t_list, ai_list, vi_list = [], [], [], [], []
    def add(X, y, t, ai=np.nan, vi=np.nan):
        X_list.append(X); y_list.append(y); t_list.append(t)
        ai_list.append(ai); vi_list.append(vi)
    for t in ["det_absent","det_auditory_only","det_visual_only"]:
        for _ in range(N_TEST_PER_TYPE):
            X, y = make_non_conflict_trial(t, rng); add(X, y, t)
    for _ in range(N_TEST_PER_TYPE):
        X, y = make_detection_conflict_trial(rng); add(X, y, "det_multisensory")
    for t in ["loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
              "loc_multisensory_same_L","loc_multisensory_same_R"]:
        for _ in range(N_TEST_PER_TYPE):
            X, y = make_non_conflict_trial(t, rng); add(X, y, t)
    for t in ["loc_conflict_audL_visR","loc_conflict_audR_visL"]:
        wins = [True]*(N_TEST_PER_TYPE//2) + [False]*(N_TEST_PER_TYPE - N_TEST_PER_TYPE//2)
        wins = list(rng.permutation(wins))
        for w in wins:
            X, y, ia, iv = make_conflict_trial_stronger_wins(t, w, rng); add(X, y, t, ia, iv)
    idx = rng.permutation(len(X_list))
    return (np.array(X_list)[idx].astype(np.float32),
            np.array(y_list)[idx].astype(np.int64),
            np.array(t_list)[idx],
            np.array(ai_list)[idx].astype(np.float32),
            np.array(vi_list)[idx].astype(np.float32))

print("Building test set...")
X_test, y_test, types_test, aud_int_test, vis_int_test = build_test_set(rng)
print("Test set:", X_test.shape, " total:", len(y_test))

## 9. Verify balance

In [ ]:
print("=== TRAINING ===  class balance:", np.bincount(y_train, minlength=4))
for t in sorted(np.unique(types_train)):
    print("  %-28s %d" % (t, np.sum(types_train==t)))
print("\n=== TEST ===  class balance:", np.bincount(y_test, minlength=4))
for t in sorted(np.unique(types_test)):
    print("  %-28s %d" % (t, np.sum(types_test==t)))
print("\nNote: det_multisensory has no true class; nominal label 1 -> accuracy reads as auditory-dominance rate.")

## 10. Visualise the dataset (bar charts)

Same 2x2 panel as v4: trial-type counts (training and test) on the top row, 4-class balance on the bottom. Training has 11 subtask types (det_multisensory is test only); the test set has all 12.

In [ ]:
# Bar charts
fig, axes = plt.subplots(2, 2, figsize=(16, 9))

train_types, train_counts = np.unique(types_train, return_counts=True)
axes[0, 0].bar(train_types, train_counts, color="steelblue", edgecolor="black")
axes[0, 0].set_title("Training set: trial type counts")
axes[0, 0].set_ylabel("Number of trials")
axes[0, 0].tick_params(axis="x", rotation=90)
for i, c in enumerate(train_counts):
    axes[0, 0].text(i, c + 15, str(c), ha="center", fontsize=9)

test_types, test_counts = np.unique(types_test, return_counts=True)
axes[0, 1].bar(test_types, test_counts, color="coral", edgecolor="black")
axes[0, 1].set_title("Test set: trial type counts")
axes[0, 1].set_ylabel("Number of trials")
axes[0, 1].tick_params(axis="x", rotation=90)
for i, c in enumerate(test_counts):
    axes[0, 1].text(i, c + 2, str(c), ha="center", fontsize=9)

class_labels = ["no det (0)", "det (1)", "right (2)", "left (3)"]
train_class_counts = [int(np.sum(y_train == c)) for c in range(4)]
axes[1, 0].bar(class_labels, train_class_counts, color="steelblue", edgecolor="black")
axes[1, 0].set_title("Training set: 4-class balance")
axes[1, 0].set_ylabel("Number of trials")
for i, c in enumerate(train_class_counts):
    axes[1, 0].text(i, c + 20, str(c), ha="center", fontsize=10)

test_class_counts = [int(np.sum(y_test == c)) for c in range(4)]
axes[1, 1].bar(class_labels, test_class_counts, color="coral", edgecolor="black")
axes[1, 1].set_title("Test set: 4-class balance")
axes[1, 1].set_ylabel("Number of trials")
for i, c in enumerate(test_class_counts):
    axes[1, 1].text(i, c + 5, str(c), ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## 11. Sanity heatmaps

In [ ]:
def plot_trial_heatmap(X, title, names=("aud_L","aud_R","vis_L","vis_R")):
    fig, ax = plt.subplots(figsize=(8, 2.4))
    im = ax.imshow(X, aspect="auto", cmap="viridis", vmin=-0.5, vmax=STIM_INTENSITY_MAX,
                   extent=[0, TRIAL_DURATION_S, N_CHANNELS-0.5, -0.5])
    ax.set_yticks(range(N_CHANNELS)); ax.set_yticklabels(names)
    ax.axvline(STIM_ONSET_S, ls="--", c="red", lw=1, alpha=0.7)
    ax.axvline(STIM_OFFSET_S, ls="--", c="red", lw=1, alpha=0.7)
    ax.set_xlabel("Time (s)"); ax.set_title(title)
    plt.colorbar(im, ax=ax, label="intensity", pad=0.01); plt.tight_layout(); plt.show()

for t in sorted(np.unique(types_test)):
    i = np.where(types_test == t)[0][0]
    plot_trial_heatmap(X_test[i], t + " (label=" + str(y_test[i]) + ")")

## 12. Save

In [ ]:
np.savez(OUT_DIR/"train.npz", X=X_train, y_4class=y_train, types=types_train,
         n_channels=N_CHANNELS, n_timesteps=N_TIMESTEPS, n_classes=4)
np.savez(OUT_DIR/"test.npz",  X=X_test,  y_4class=y_test,  types=types_test,
         aud_int=aud_int_test, vis_int=vis_int_test,
         n_channels=N_CHANNELS, n_timesteps=N_TIMESTEPS, n_classes=4)
print("Saved to", OUT_DIR.absolute())
for f in sorted(OUT_DIR.iterdir()):
    print("  ", f.name, "(%.0f KB)" % (f.stat().st_size/1024))

## 13. Summary

**Training: 4800 trials, balanced 1200 per class.** det_multisensory excluded.
**Test: 1200 trials, 100 per type, all 12 subtasks.**

Two readouts at test:
- **Detection conflict** (`det_multisensory`): "detected" (class 1, auditory dominance) vs "no detection" (class 0, visual dominance)?
- **Localisation conflict** (`loc_conflict_*`): follows the stronger modality (additive) or a fixed bias?